# 2. 회귀모형 안정성 점검

## 목적

최종농도 설명모형에서 산 총투입량과 후기 기질 기울기의 중요성이 변수 제외, Ridge 규제, 영향 배치 제외, 전략별 분석에서도 유지되는지 확인한다. 분석 단위는 정상 배치 90개이며 CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    late = time / time[-1] >= 0.8
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    substrate = batch['기질농도(g/L)'].to_numpy()
    rows.append({
        '배치번호': batch_number, '전략': 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC'),
        '최종농도': penicillin[-1],
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '기질후기기울기': np.polyfit(time[late], substrate[late], 1)[0],
        'OUR평균': batch['산소소모율(g/min)'].mean(),
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO평균': batch['용존산소(mg/L)'].mean(),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.groupby('전략').size().rename('배치수').to_frame())

,배치수
전략,
APC,30
OC,30
RC,30


### 판단

RC·OC·APC 각각 30개 배치를 사용한다. 후기 기질 기울기는 실제 종료시간을 이용한 종료 후 설명변수이므로 실시간 예측변수로 해석하지 않는다.

In [2]:
all_features = ['산총투입량', '기질후기기울기', 'OUR평균', 'pH표준편차', 'DO평균']
model_features = {
    '전체모형': all_features,
    '산투입제외': [x for x in all_features if x != '산총투입량'],
    '기질기울기제외': [x for x in all_features if x != '기질후기기울기'],
    '두변수제외': ['OUR평균', 'pH표준편차', 'DO평균'],
}
rng = np.random.default_rng(42)
fold_ids = np.empty(len(batch_metrics), dtype=int)
for strategy in ['RC', 'OC', 'APC']:
    indices = np.flatnonzero(batch_metrics['전략'].eq(strategy).to_numpy())
    rng.shuffle(indices)
    fold_ids[indices] = np.arange(len(indices)) % 5


def make_design(train_mask, test_mask, features):
    mean = batch_metrics.loc[train_mask, features].mean().to_numpy()
    std = batch_metrics.loc[train_mask, features].std(ddof=1).replace(0, 1).to_numpy()
    def build(mask):
        return np.column_stack([
            np.ones(mask.sum()),
            batch_metrics.loc[mask, '전략'].eq('OC').astype(float),
            batch_metrics.loc[mask, '전략'].eq('APC').astype(float),
            (batch_metrics.loc[mask, features].to_numpy() - mean) / std,
        ])
    return build(train_mask), build(test_mask)


target = batch_metrics['최종농도'].to_numpy()
comparison_rows = []
for model_name, features in model_features.items():
    predictions = np.zeros(len(target))
    for fold in range(5):
        train = fold_ids != fold
        test = ~train
        train_design, test_design = make_design(train, test, features)
        predictions[test] = test_design @ (np.linalg.pinv(train_design) @ target[train])
    all_rows = np.ones(len(target), dtype=bool)
    full_design, _ = make_design(all_rows, np.zeros(len(target), dtype=bool), features)
    fitted = full_design @ (np.linalg.pinv(full_design) @ target)
    r_squared = 1 - np.sum((target - fitted) ** 2) / np.sum((target - target.mean()) ** 2)
    adjusted_r_squared = 1 - (1 - r_squared) * (len(target) - 1) / (len(target) - full_design.shape[1])
    cv_r_squared = 1 - np.sum((target - predictions) ** 2) / np.sum((target - target.mean()) ** 2)
    comparison_rows.append({
        '모형': model_name, 'R2': r_squared, '수정R2': adjusted_r_squared,
        '교차검증R2': cv_r_squared, '교차검증RMSE': np.sqrt(np.mean((target - predictions) ** 2)),
    })
model_comparison = pd.DataFrame(comparison_rows)
display(model_comparison.round(6))

,모형,R2,수정R2,교차검증R2,교차검증RMSE
0,전체모형,0.895109,0.886154,0.855595,2.899989
1,산투입제외,0.822787,0.809977,0.778428,3.592227
2,기질기울기제외,0.876849,0.867946,0.840483,3.047960
3,두변수제외,0.532942,0.505141,0.443400,5.693476


### 판단

전체모형 교차검증 R²는 0.856이다. 산 총투입량을 제외하면 0.778, 후기 기질 기울기를 제외하면 0.840, 둘 다 제외하면 0.443으로 감소한다. 산 총투입량이 더 큰 설명력을 가지며, 두 변수는 서로 중복되지만 각각 추가 정보를 제공한다.

In [3]:
ridge_rows = []
for alpha in [0.01, 0.1, 1, 10, 100]:
    predictions = np.zeros(len(target))
    for fold in range(5):
        train = fold_ids != fold
        test = ~train
        train_design, test_design = make_design(train, test, all_features)
        penalty = np.eye(train_design.shape[1])
        penalty[0, 0] = 0
        coefficients = np.linalg.solve(train_design.T @ train_design + alpha * penalty, train_design.T @ target[train])
        predictions[test] = test_design @ coefficients
    ridge_rows.append({
        'alpha': alpha,
        '교차검증R2': 1 - np.sum((target - predictions) ** 2) / np.sum((target - target.mean()) ** 2),
        '교차검증RMSE': np.sqrt(np.mean((target - predictions) ** 2)),
    })
ridge_results = pd.DataFrame(ridge_rows)
display(ridge_results.round(6))

all_rows = np.ones(len(target), dtype=bool)
design, _ = make_design(all_rows, np.zeros(len(target), dtype=bool), all_features)
term_names = ['절편', 'OC-RC', 'APC-RC'] + all_features
ols_coefficients = np.linalg.pinv(design) @ target
penalty = np.eye(design.shape[1])
penalty[0, 0] = 0
ridge_coefficients = np.linalg.solve(design.T @ design + 1.0 * penalty, design.T @ target)
display(pd.DataFrame({'변수': term_names, 'OLS계수': ols_coefficients, 'Ridge_alpha1계수': ridge_coefficients}).round(6))

,alpha,교차검증R2,교차검증RMSE
0,0.01,0.855638,2.899562
1,0.10,0.856010,2.895823
2,1.00,0.858854,2.867082
3,10.00,0.858686,2.868793
4,100.00,0.726288,3.992575


,변수,OLS계수,Ridge_alpha1계수
0,절편,25.200829,25.183340
1,OC-RC,0.585573,0.531784
2,APC-RC,-0.957879,-0.851624
3,산총투입량,-5.272403,-5.032339
4,기질후기기울기,-2.470672,-2.557125
5,OUR평균,-0.147721,-0.037313
6,pH표준편차,-0.710443,-0.715260
7,DO평균,-0.377110,-0.348971


### 판단

Ridge는 alpha=1에서 교차검증 R² 0.859로 가장 높아 전체 OLS보다 소폭 개선됐다. alpha=1에서도 산 총투입량 계수는 -5.03, 후기 기질 기울기는 -2.56으로 음의 방향과 상대적 중요성이 유지됐다. 따라서 다중공선성이 핵심 결론을 뒤집지는 않는다.

In [4]:
residuals = target - design @ ols_coefficients
n_rows, n_parameters = design.shape
mse = np.sum(residuals ** 2) / (n_rows - n_parameters)
inverse_xtx = np.linalg.pinv(design.T @ design)
leverage = np.sum(design * (design @ inverse_xtx), axis=1)
cooks_distance = (residuals ** 2 / (n_parameters * mse)) * leverage / (1 - leverage) ** 2
keep = cooks_distance <= 4 / n_rows
refit_coefficients = np.linalg.pinv(design[keep]) @ target[keep]
influential = batch_metrics.loc[~keep, ['배치번호', '전략', '최종농도']].copy()
influential['CookD'] = cooks_distance[~keep]
print(f'CookD > 4/n 영향 배치: {(~keep).sum()}개')
display(influential.round(6))
display(pd.DataFrame({
    '변수': term_names, '전체계수': ols_coefficients,
    '영향배치제외계수': refit_coefficients, '변화량': refit_coefficients - ols_coefficients,
}).round(6))

strategy_rows = []
for strategy in ['RC', 'OC', 'APC']:
    subset = batch_metrics.loc[batch_metrics['전략'].eq(strategy)]
    standardized = (subset[all_features] - subset[all_features].mean()) / subset[all_features].std(ddof=1)
    strategy_design = np.column_stack([np.ones(len(subset)), standardized])
    coefficients = np.linalg.pinv(strategy_design) @ subset['최종농도'].to_numpy()
    strategy_rows.append({'전략': strategy, **dict(zip(all_features, coefficients[1:]))})
display(pd.DataFrame(strategy_rows).round(6))

CookD > 4/n 영향 배치: 9개


,배치번호,전략,최종농도,CookD
2,3,RC,17.428,0.179981
6,7,RC,29.416,0.057318
32,33,OC,13.186,0.054614
35,36,OC,13.717,0.055123
38,39,OC,27.977,0.120440
43,44,OC,12.672,0.491659
47,48,OC,34.756,0.046867
59,60,OC,18.818,0.054275
66,67,APC,21.596,0.259204


,변수,전체계수,영향배치제외계수,변화량
0,절편,25.200829,24.394222,-0.806607
1,OC-RC,0.585573,1.089307,0.503734
2,APC-RC,-0.957879,-0.230570,0.727309
3,산총투입량,-5.272403,-6.125411,-0.853008
4,기질후기기울기,-2.470672,-1.750615,0.720057
5,OUR평균,-0.147721,-0.824730,-0.677009
6,pH표준편차,-0.710443,-2.643590,-1.933147
7,DO평균,-0.377110,-1.083871,-0.706762


,전략,산총투입량,기질후기기울기,OUR평균,pH표준편차,DO평균
0,RC,-6.737777,-1.756547,-0.977972,-0.050821,-1.682028
1,OC,-5.042880,-1.971524,1.237272,0.430359,0.882817
2,APC,-0.719058,0.326088,-0.309828,-1.184764,-0.914801


### 최종 판단과 결론

- Cook 기준 영향 배치 9개를 제외해도 산 총투입량 계수는 -5.27에서 -6.13, 후기 기질 기울기는 -2.47에서 -1.75로 모두 음의 방향을 유지한다.
- 산 총투입량 계수는 RC·OC·APC 모두 음수여서 가장 안정적인 저성과 신호다.
- 후기 기질 기울기는 RC와 OC에서는 음수지만 APC에서는 +0.33으로 방향이 바뀐다. APC는 기질 값이 거의 0에 몰려 변동폭이 작기 때문에 전체 관계를 전략별 원인으로 일반화하면 안 된다.
- 최종 결론은 산 총투입량의 음의 관계는 강건하고, 후기 기질 기울기는 전체 예측에는 유용하지만 전략별 안정성은 부족하다는 것이다. 모형은 설명·예측용이며 인과효과나 현장 조작 기준을 직접 제공하지 않는다.